# TD 5 — Querying your documents: RAG 📚

An LLM does not know **your** documents. The most common method to help it is called **RAG** (*Retrieval-Augmented Generation*):

1. **split** the documents into passages;
2. **search** for the passages close to the question;
3. **give** these passages to the model, with the question;
4. **check** that the answer really relies on them.

**At the end of this lab, you will know** how these four steps work, without a specialised library, and how to measure their quality.

> Our corpus is tiny: six stories, about 2,000 words. It would fit entirely in the context window. This is perfect to **understand**, and we will compare at the end. RAG becomes essential with thousands of documents.

## Set up the environment ⚙️

This lab works with a **local** model (Ollama, see TD 2) or a **hosted** one (Gemini, see TD 1). Choose in the next cell. Packages needed: `pydantic numpy pandas scikit-learn`, + `ollama` / `google-genai python-dotenv`.

The `ask` function sends a prompt and returns the text of the answer. If you give it a Pydantic schema, the answer is JSON.

> A local model with 3 billion parameters makes mistakes: this is normal, and interesting to observe. Do not use FunctionGemma here, it is not made for reading texts.

In [ ]:
import os
import json
from pathlib import Path
from time import sleep

BACKEND = "ollama"            # "ollama" (local) or "gemini" (hosted)
MODELS = {"ollama": "qwen2.5:3b", "gemini": "gemini-3.5-flash-lite"}

def ask(prompt, schema=None, model=None, backend=None):
    """Send a prompt, return the text of the answer (JSON if a schema is given)."""
    backend = backend or BACKEND
    model = model or MODELS[backend]

    if backend == "ollama":
        import ollama
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            format=schema.model_json_schema() if schema else None,
            options={"temperature": 0, "num_ctx": 8192, "num_predict": 3000},
        )
        return response.message.content

    from dotenv import load_dotenv
    from google import genai
    from google.genai import errors, types
    load_dotenv()
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    config = types.GenerateContentConfig(
        response_mime_type="application/json" if schema else None,
        response_schema=schema,
    )
    for attempt in range(4):
        try:
            return client.models.generate_content(model=model, contents=prompt, config=config).text
        except errors.APIError as error:
            if error.code not in (429, 500, 503) or attempt == 3:
                raise
            sleep(2 ** attempt)

print(ask("Say hello in French, in three words."))

## 1. Split ✂️

A passage must be short enough to be precise, and long enough to be understood alone. Here, the stories are already divided into paragraphs: a good natural way to split.

### Demonstration: create the passages 📄

Each passage is a dictionary with the keys `id` (0, 1, 2...), `source` (file name) and `text`. A paragraph is a non-empty line. We ignore the lines of fewer than 8 words (greetings, signatures).

In [ ]:
passages = []
for path in sorted(Path("data/texts").glob("*.txt")):
    for line in path.read_text(encoding="utf-8").splitlines():
        if len(line.split()) >= 8:
            passages.append({"id": len(passages), "source": path.name, "text": line.strip()})

print(len(passages), "passages")
print(passages[0])

## 2. Search with words 🔤

The classic method, **TF-IDF**, compares words. A word matters more if it is frequent in the passage and rare in the others. Each passage becomes a vector, and we compare vectors with the **cosine similarity**:

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

`1`: same direction. `0`: nothing in common.

### Demonstration: a search with words 🔎

The `cosine` function computes the similarity. Let's check it: perpendicular directions give 0, the same direction gives 1.

Then `search_words(question, k=3)` returns the `k` closest passages, according to scikit-learn's `TfidfVectorizer`. We ask two questions: "Où a-t-on vu des ammonites ?" (where were ammonites seen?) and "Quel voyageur a perdu son couvre-chef ?" (which traveller lost their headgear?).

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / norm) if norm else 0.0

print(cosine([1, 0], [0, 1]), round(cosine([1, 2], [2, 4]), 6))

vectorizer = TfidfVectorizer(strip_accents="unicode")
word_vectors = vectorizer.fit_transform(p["text"] for p in passages).toarray()

def top(scores, k):
    order = np.argsort(scores)[::-1][:k]
    return [{**passages[i], "score": round(float(scores[i]), 3)} for i in order]

def search_words(question, k=3):
    q = vectorizer.transform([question]).toarray()[0]
    return top([cosine(q, v) for v in word_vectors], k)

for question in ("Où a-t-on vu des ammonites ?", "Quel voyageur a perdu son couvre-chef ?"):
    print(question)
    for p in search_words(question):
        print("  ", p["score"], p["source"], "|", p["text"][:80])

- Question 1: the word "on" weighs more than "ammonites", and the right passage does not come first!
- Question 2: "couvre-chef" never appears, the texts say "chapeau" (hat). The right passage is first only because it contains "perdu" (lost).

Words are not meaning.

## 3. Search with meaning 🧭

An **embedding model** turns a text into a vector, so that two texts with a close **meaning** have close vectors, even with different words.

With Ollama, download `embeddinggemma:300m` (about 0.6 GB), a multilingual model:

```bash
ollama pull embeddinggemma:300m
```

These models often expect a short prefix, which separates **questions** from **documents**. The `embed` function takes care of it.

In [ ]:
def embed(texts, is_question=False):
    """Turn a list of texts into a list of vectors."""
    prefix = "task: search result | query: " if is_question else "title: none | text: "
    texts = [prefix + text for text in texts]

    if BACKEND == "ollama":
        import ollama
        return np.array(ollama.embed(model="embeddinggemma:300m", input=texts).embeddings)

    from dotenv import load_dotenv
    from google import genai
    load_dotenv()
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    return np.array([
        client.models.embed_content(model="gemini-embedding-2", contents=text).embeddings[0].values
        for text in texts
    ])

words = ["chapeau", "casquette", "boulangerie", "montagne"]
vectors = embed(words)
print(vectors.shape)

### Demonstration: a search with meaning 🧲

Let's first compare "chapeau" (hat) with the three other words. Then we compute the vectors of all the passages **only once**. The `search_meaning(question, k=3)` function then only computes the vector of the question. Let's ask the same two questions again.

In [ ]:
for word, vector in zip(words[1:], vectors[1:]):
    print(f"chapeau / {word:12s} {cosine(vectors[0], vector):.3f}")

passage_vectors = embed([p["text"] for p in passages])

def search_meaning(question, k=3):
    q = embed([question], is_question=True)[0]
    return top([cosine(q, v) for v in passage_vectors], k)

for question in ("Où a-t-on vu des ammonites ?", "Quel voyageur a perdu son couvre-chef ?"):
    print(question)
    for p in search_meaning(question):
        print("  ", p["score"], p["source"], "|", p["text"][:80])

## 4. Measure the search 📏

If the right passage is not found, the model cannot answer. So we measure the search **separately**: for each question, is the right passage among the first `k`? This is the **recall at k** (*recall@k*).

For each question, we give a few words that only appear in the right passage.

In [ ]:
tests = [
    ("Qui a partagé son repas avec un oiseau ?", "canard"),
    ("Où la voyageuse a-t-elle parlé de parfums ?", "discuter parfums"),
    ("Quel voyageur a retrouvé son couvre-chef dans son sac ?", "coincé au fond du sac"),
    ("Où le vent a-t-il empêché de manger au sommet ?", "renoncé à mon pique-nique"),
    ("Quel moyen de transport a fait du bruit sur l'île ?", "grinçait"),
    ("Dans quelle ville a-t-on suivi les conseils d'un vendeur de livres ?", "Un libraire"),
    ("Où peut-on voir des fossiles ?", "ammonites"),
    ("Quelle sœur aimait comparer les pains ?", "enquête comparative"),
]

### Demonstration: compare both searches ⚖️

The `recall_at_k(search, k)` function computes the share of questions whose right passage is among the first `k`. Let's compare both searches for k = 1, 3 and 5, then display the failed questions, if any.

In [ ]:
import pandas as pd

def recall_at_k(search, k):
    hits = [any(clue in p["text"] for p in search(question, k)) for question, clue in tests]
    return sum(hits) / len(hits)

print(pd.DataFrame(
    {f"k={k}": [recall_at_k(search_words, k), recall_at_k(search_meaning, k)] for k in (1, 3, 5)},
    index=["words", "meaning"],
))

for question, clue in tests:
    if not any(clue in p["text"] for p in search_meaning(question, 3)):
        print("Failed:", question)

There are only eight questions: read this table as an illustration, not as a proof (see TD 4).

## 5. Generate the answer ✍️

We give the model the numbered passages, with three rules:

- answer **only** with these passages;
- **quote** the sentence used;
- say that it does not know if the answer is not there.

### Demonstration: ask the corpus a question 💬

The `Answer` schema contains `found` (is the answer in the passages?), `answer`, `passage_id` and `quote`. The `rag(question, k=3)` function searches for the passages, builds the prompt and returns an `Answer` object.

The answer to the first question is in the texts. The price of the ticket to Porquerolles is not.

In [ ]:
from pydantic import BaseModel

class Answer(BaseModel):
    found: bool
    answer: str
    passage_id: int | None
    quote: str | None

def build_prompt(question, selected):
    context = "\n\n".join(f"[{p['id']}] {p['text']}" for p in selected)
    return f"""Answer the question using only the passages below.
- If the answer is not in the passages: found = false, passage_id = null, quote = null.
- Otherwise: found = true, passage_id = the number of the passage used,
  quote = the exact words of the passage that prove the answer.
- answer = a complete sentence in French that replies to the question (who, where, what),
  or "Je ne sais pas." if found is false.

Passages:
{context}

Question: {question}"""

def rag(question, k=3):
    return Answer.model_validate_json(ask(build_prompt(question, search_meaning(question, k)), schema=Answer))

for question in ("Où la voyageuse a-t-elle parlé de parfums ?", "Quel est le prix du billet pour Porquerolles ?"):
    print(question, "\n  ", rag(question))

## 6. Check the quote 🔎

A quote can be invented. Fortunately, **a program can check it**: the quoted sentence must be, word for word, in the passage given.

### Demonstration: detect invented quotes 🕵️

The `check(answer)` function returns:

- `"no answer"` if `found` is false;
- `"verified"` if the quote is in the passage given;
- `"wrong passage"` if it is in another passage;
- `"invented"` otherwise.

It ignores differences in capital letters, apostrophes and spaces. An empty quote is not evidence.

In [ ]:
import re

def clean(text):
    return re.sub(r"\s+", " ", text.lower().replace("’", "'")).strip(' ."«»')

def check(answer):
    if not answer.found:
        return "no answer"
    quote = clean(answer.quote or "")
    if len(quote) < 5:
        return "invented"
    matches = [p["id"] for p in passages if quote in clean(p["text"])]
    if answer.passage_id in matches:
        return "verified"
    return "wrong passage" if matches else "invented"

for question, _ in tests:
    answer = rag(question)
    print(f"{check(answer):14s} {question}\n{'':14s} -> {answer.answer}")

A small model sometimes contradicts itself: it claims that it found the answer, then says "Je ne sais pas" (I don't know). The check catches it.

A verified quote does not prove that the answer is right: it can be misread, or support another conclusion. But an invented quote is a good reason to reject the answer.

## 7. What if we gave everything? 🪣

Our corpus is small: we can send all of it. Is RAG still useful?

### Demonstration: RAG versus the whole corpus ⚖️

The `ask_everything(question)` function gives **all** the passages to the model. For three questions, let's compare it with `rag`: length of the prompt, time, result of `check` and answer.

In [ ]:
from time import perf_counter

def ask_everything(question):
    return Answer.model_validate_json(ask(build_prompt(question, passages), schema=Answer))

for question, _ in tests[:3]:
    print(question)
    print("   words in the prompt: RAG", len(build_prompt(question, search_meaning(question)).split()),
          "/ everything", len(build_prompt(question, passages).split()))
    for function in (rag, ask_everything):
        start = perf_counter()
        answer = function(question)
        print(f"   {function.__name__:15s} {perf_counter() - start:5.1f} s  {check(answer):14s} {answer.answer}")

The complete prompt is about ten times longer, so it is slower to read. With thousands of documents, it would no longer fit in the context window at all.

## Discussion 💭

- "How many travellers took the train?" Why does RAG answer badly when you have to **count** in the whole corpus?
- What happens if two documents contradict each other?
- And what if a document contains a **hidden instruction** for the model? This is the topic of TD 6.
- Are you allowed to send your documents to an online service (GDPR, copyright, confidentiality)?

## Key points 📋

| Step | How to check it |
| --- | --- |
| Splitting | read a few passages: can they be understood alone? |
| Search | recall at k |
| Generation | compare with expected answers, including "I don't know" |
| Quote | exact comparison with the passage |